# EDA Features + Random Forest Feature Selection

Controlled experiment:

1. Start from the same baseline data.
2. Add 10 EDA-based features.
3. Preprocess with the same scaler + one-hot strategy.
4. Use Random Forest feature importance on train only.
5. Select top K encoded features.
6. Retrain Random Forest and SVM.
7. Compare with baseline and +10 EDA features.

In [1]:
from pathlib import Path
import time
import warnings

import joblib
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.svm import LinearSVC

warnings.filterwarnings("ignore")

ROOT_DIR = Path.cwd().parents[1] if Path.cwd().name == "feature_engineering" else Path.cwd()
DATA_PATH = ROOT_DIR / "data" / "diabetic_data_clean_common.csv"
BASELINE_RANKING_PATH = ROOT_DIR / "train" / "model_comparison" / "outputs" / "practical_model_ranking.csv"
EDA_10_METRICS_PATH = ROOT_DIR / "train" / "feature_engineering" / "eda_10_features_outputs" / "validation_metrics.csv"
OUTPUT_DIR = ROOT_DIR / "train" / "feature_engineering" / "eda_10_rf_selection_outputs"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

TARGET_COLUMN = "readmitted_binary"
RANDOM_STATE = 42

DATA_PATH, BASELINE_RANKING_PATH, EDA_10_METRICS_PATH, OUTPUT_DIR

(WindowsPath('d:/HocTap/KT&XLTT/CUOIKI/data/diabetic_data_clean_common.csv'),
 WindowsPath('d:/HocTap/KT&XLTT/CUOIKI/train/model_comparison/outputs/practical_model_ranking.csv'),
 WindowsPath('d:/HocTap/KT&XLTT/CUOIKI/train/feature_engineering/eda_10_features_outputs/validation_metrics.csv'),
 WindowsPath('d:/HocTap/KT&XLTT/CUOIKI/train/feature_engineering/eda_10_rf_selection_outputs'))

## 1. Load And Split Data

Use the same split strategy as baseline and EDA-10 notebooks.

In [2]:
df = pd.read_csv(DATA_PATH)

y = df[TARGET_COLUMN]
X = df.drop(columns=[TARGET_COLUMN])

X_train_val, X_test, y_train_val, y_test = train_test_split(
    X, y, test_size=0.20, stratify=y, random_state=RANDOM_STATE
)

X_train, X_val, y_train, y_val = train_test_split(
    X_train_val, y_train_val, test_size=0.20, stratify=y_train_val, random_state=RANDOM_STATE
)

print("X_train:", X_train.shape)
print("X_val:", X_val.shape)
print("X_test:", X_test.shape)

X_train: (65128, 50)
X_val: (16282, 50)
X_test: (20353, 50)


## 2. Add 10 EDA-Based Features

In [3]:
DRUG_COLUMNS = [
    "metformin", "repaglinide", "nateglinide", "chlorpropamide", "glimepiride",
    "acetohexamide", "glipizide", "glyburide", "tolbutamide", "pioglitazone",
    "rosiglitazone", "acarbose", "miglitol", "troglitazone", "tolazamide",
    "insulin", "glyburide-metformin", "glipizide-metformin",
    "glimepiride-pioglitazone", "metformin-rosiglitazone", "metformin-pioglitazone",
]


def safe_divide(numerator, denominator):
    denominator = denominator.replace(0, np.nan)
    return (numerator / denominator).replace([np.inf, -np.inf], np.nan).fillna(0)


def fit_fe_params(X_train):
    return {"q3_time_in_hospital": X_train["time_in_hospital"].quantile(0.75)}


def add_10_eda_features(X, params):
    X_fe = X.copy()
    X_fe["total_prior_visits"] = X_fe["number_outpatient"] + X_fe["number_emergency"] + X_fe["number_inpatient"]
    X_fe["has_prior_inpatient"] = (X_fe["number_inpatient"] > 0).astype(int)
    X_fe["meds_per_day"] = safe_divide(X_fe["num_medications"], X_fe["time_in_hospital"])
    X_fe["labs_per_day"] = safe_divide(X_fe["num_lab_procedures"], X_fe["time_in_hospital"])
    X_fe["is_long_stay"] = (X_fe["time_in_hospital"] >= params["q3_time_in_hospital"]).astype(int)
    X_fe["is_senior"] = (X_fe["age_ordinal"] >= 6).astype(int)
    X_fe["a1c_abnormal"] = X_fe["A1Cresult"].fillna("None").isin([">7", ">8"]).astype(int)
    X_fe["insulin_changed"] = X_fe["insulin"].isin(["Up", "Down"]).astype(int)
    X_fe["num_diabetes_drugs_used"] = X_fe[DRUG_COLUMNS].fillna("No").ne("No").sum(axis=1)
    X_fe["num_unique_diag_groups"] = X_fe[["diag_1_group", "diag_2_group", "diag_3_group"]].fillna("Unknown").nunique(axis=1)
    return X_fe


fe_params = fit_fe_params(X_train)
X_train_fe = add_10_eda_features(X_train, fe_params)
X_val_fe = add_10_eda_features(X_val, fe_params)
X_test_fe = add_10_eda_features(X_test, fe_params)

new_features = sorted(set(X_train_fe.columns) - set(X_train.columns))
pd.DataFrame({"new_feature": new_features})

,new_feature
0,a1c_abnormal
1,has_prior_inpatient
2,insulin_changed
3,is_long_stay
4,is_senior
5,labs_per_day
6,meds_per_day
7,num_diabetes_drugs_used
8,num_unique_diag_groups
9,total_prior_visits


## 3. Preprocess

Same preprocessing style: scale numeric columns and one-hot encode categorical columns.

In [4]:
DROP_COLUMNS = ["readmitted", "diag_1", "diag_2", "diag_3", "age", "age_midpoint"]

X_train_model = X_train_fe.drop(columns=[col for col in DROP_COLUMNS if col in X_train_fe.columns])
X_val_model = X_val_fe.drop(columns=[col for col in DROP_COLUMNS if col in X_val_fe.columns])
X_test_model = X_test_fe.drop(columns=[col for col in DROP_COLUMNS if col in X_test_fe.columns])

categorical_cols = X_train_model.select_dtypes(include=["object", "category"]).columns.tolist()
numeric_cols = [col for col in X_train_model.columns if col not in categorical_cols]

print("Input columns before encoding:", X_train_model.shape[1])
print("Numeric columns:", len(numeric_cols))
print("Categorical columns:", len(categorical_cols))

Input columns before encoding: 54
Numeric columns: 22
Categorical columns: 32


In [5]:
def make_one_hot_encoder():
    try:
        return OneHotEncoder(handle_unknown="ignore", sparse_output=False)
    except TypeError:
        return OneHotEncoder(handle_unknown="ignore", sparse=False)


preprocessor = ColumnTransformer(
    transformers=[
        ("num", Pipeline([("scaler", StandardScaler())]), numeric_cols),
        ("cat", Pipeline([("onehot", make_one_hot_encoder())]), categorical_cols),
    ]
)

X_train_processed = preprocessor.fit_transform(X_train_model)
X_val_processed = preprocessor.transform(X_val_model)
X_test_processed = preprocessor.transform(X_test_model)
feature_names = preprocessor.get_feature_names_out()

print("Processed X_train:", X_train_processed.shape)
print("Processed X_val:", X_val_processed.shape)
print("Processed X_test:", X_test_processed.shape)

Processed X_train: (65128, 253)
Processed X_val: (16282, 253)
Processed X_test: (20353, 253)


## 4. Random Forest Feature Selection

Fit the selector on train only. Then apply the selected encoded feature indices to validation and test.

In [6]:
selector = RandomForestClassifier(
    n_estimators=200,
    max_depth=None,
    min_samples_leaf=10,
    class_weight="balanced_subsample",
    n_jobs=-1,
    random_state=RANDOM_STATE,
)

start_time = time.time()
selector.fit(X_train_processed, y_train)
print(f"Feature selector trained in {time.time() - start_time:.2f} seconds")

importance_df = pd.DataFrame({
    "feature": feature_names,
    "importance": selector.feature_importances_,
}).sort_values("importance", ascending=False).reset_index(drop=True)

importance_df.head(30)

Feature selector trained in 4.51 seconds


,feature,importance
0,num__total_prior_visits,0.085386
1,num__number_inpatient,0.081112
2,num__has_prior_inpatient,0.058296
3,num__labs_per_day,0.048683
4,num__num_lab_procedures,0.047836
5,num__num_medications,0.045416
6,num__meds_per_day,0.045230
7,num__discharge_disposition_id,0.044751
8,num__number_diagnoses,0.039151
9,num__time_in_hospital,0.027297


## 5. Retrain Models On Top K Features

In [7]:
TOP_K_VALUES = [50, 75, 100, 150]


def build_models():
    return {
        "Random Forest": RandomForestClassifier(
            n_estimators=200,
            max_depth=None,
            min_samples_leaf=10,
            class_weight="balanced_subsample",
            n_jobs=-1,
            random_state=RANDOM_STATE,
        ),
        "SVM": LinearSVC(
            C=1.0,
            class_weight="balanced",
            max_iter=5000,
            random_state=RANDOM_STATE,
        ),
    }


def get_score_for_auc(model, X):
    if hasattr(model, "predict_proba"):
        return model.predict_proba(X)[:, 1]
    if hasattr(model, "decision_function"):
        return model.decision_function(X)
    return None


def evaluate_model(model, X_val_selected):
    y_pred = model.predict(X_val_selected)
    y_score = get_score_for_auc(model, X_val_selected)
    return {
        "accuracy": accuracy_score(y_val, y_pred),
        "precision": precision_score(y_val, y_pred, zero_division=0),
        "recall": recall_score(y_val, y_pred, zero_division=0),
        "f1_score": f1_score(y_val, y_pred, zero_division=0),
        "roc_auc": roc_auc_score(y_val, y_score) if y_score is not None else None,
    }, y_pred


results = []
trained_models = {}
confusion_matrices = {}
classification_reports = {}
selected_feature_sets = {}

for top_k in TOP_K_VALUES:
    selected_idx = importance_df.head(top_k).index.to_numpy()
    selected_features = importance_df.head(top_k)["feature"].tolist()
    selected_feature_sets[f"top_{top_k}"] = selected_features

    X_train_selected = X_train_processed[:, selected_idx]
    X_val_selected = X_val_processed[:, selected_idx]

    for model_name, model in build_models().items():
        print(f"Training {model_name} with top_{top_k} features...")
        start_time = time.time()
        model.fit(X_train_selected, y_train)
        train_time = time.time() - start_time

        metrics, y_pred = evaluate_model(model, X_val_selected)
        row = {
            "model": model_name,
            "feature_set": f"top_{top_k}",
            "num_features": top_k,
            **metrics,
            "train_time_sec": train_time,
        }
        results.append(row)

        model_key = f"{model_name}__top_{top_k}"
        trained_models[model_key] = model
        confusion_matrices[model_key] = pd.DataFrame(
            confusion_matrix(y_val, y_pred),
            index=["actual_0", "actual_1"],
            columns=["predicted_0", "predicted_1"],
        )
        classification_reports[model_key] = pd.DataFrame(
            classification_report(y_val, y_pred, output_dict=True, zero_division=0)
        ).T

        print(
            f"Done: F1={metrics['f1_score']:.4f}, Recall={metrics['recall']:.4f}, "
            f"Precision={metrics['precision']:.4f}, Accuracy={metrics['accuracy']:.4f}, "
            f"ROC-AUC={metrics['roc_auc']:.4f}"
        )

selection_results_df = pd.DataFrame(results).sort_values(
    ["f1_score", "roc_auc", "accuracy"], ascending=False
).reset_index(drop=True)

selection_results_df

Training Random Forest with top_50 features...
Done: F1=0.5958, Recall=0.5830, Precision=0.6092, Accuracy=0.6354, ROC-AUC=0.6923
Training SVM with top_50 features...
Done: F1=0.5588, Recall=0.5217, Precision=0.6017, Accuracy=0.6204, ROC-AUC=0.6577
Training Random Forest with top_75 features...
Done: F1=0.5971, Recall=0.5802, Precision=0.6151, Accuracy=0.6392, ROC-AUC=0.6936
Training SVM with top_75 features...
Done: F1=0.5639, Recall=0.5317, Precision=0.6003, Accuracy=0.6210, ROC-AUC=0.6587
Training Random Forest with top_100 features...
Done: F1=0.5965, Recall=0.5808, Precision=0.6132, Accuracy=0.6379, ROC-AUC=0.6947
Training SVM with top_100 features...
Done: F1=0.5645, Recall=0.5340, Precision=0.5988, Accuracy=0.6203, ROC-AUC=0.6599
Training Random Forest with top_150 features...
Done: F1=0.5975, Recall=0.5804, Precision=0.6157, Accuracy=0.6397, ROC-AUC=0.6956
Training SVM with top_150 features...
Done: F1=0.5699, Recall=0.5450, Precision=0.5972, Accuracy=0.6209, ROC-AUC=0.6629


,model,feature_set,num_features,accuracy,precision,recall,f1_score,roc_auc,train_time_sec
0,Random Forest,top_150,150,0.639663,0.615722,0.580357,0.597517,0.695611,3.388638
1,Random Forest,top_75,75,0.639172,0.615059,0.580224,0.597134,0.693607,3.563964
2,Random Forest,top_100,100,0.637944,0.613198,0.580757,0.596537,0.694691,3.383751
3,Random Forest,top_50,50,0.635426,0.609162,0.583022,0.595806,0.692347,2.770504
4,SVM,top_150,150,0.620870,0.597167,0.545043,0.569916,0.662942,6.389198
5,SVM,top_100,100,0.620317,0.598775,0.533982,0.564525,0.659892,3.724898
6,SVM,top_75,75,0.620993,0.600271,0.531716,0.563918,0.658701,3.978092
7,SVM,top_50,50,0.620378,0.601660,0.521722,0.558847,0.657708,1.178305


## 6. Compare With Baseline And +10 EDA Features

In [8]:
baseline_df = pd.read_csv(BASELINE_RANKING_PATH)
baseline_df = baseline_df[baseline_df["model"].isin(["Random Forest", "SVM"])].copy()
baseline_df["version"] = "Baseline"
baseline_df["feature_set"] = "all_baseline_features"
baseline_df["num_features"] = np.nan

comparison_frames = [baseline_df]

if EDA_10_METRICS_PATH.exists():
    eda10_df = pd.read_csv(EDA_10_METRICS_PATH)
    eda10_df = eda10_df[eda10_df["model"].isin(["Random Forest", "SVM"])].copy()
    eda10_df["version"] = "+10 EDA features"
    eda10_df["feature_set"] = "all_eda_10_features"
    eda10_df["num_features"] = np.nan
    comparison_frames.append(eda10_df)

selection_compare_df = selection_results_df.copy()
selection_compare_df["version"] = "+10 EDA features + RF selection"
comparison_frames.append(selection_compare_df)

comparison_cols = [
    "model", "version", "feature_set", "num_features",
    "accuracy", "precision", "recall", "f1_score", "roc_auc",
]
all_comparison_df = pd.concat(
    [frame[comparison_cols] for frame in comparison_frames],
    ignore_index=True,
)

best_selection_df = selection_results_df.head(1).copy()
best_model_key = f"{best_selection_df.loc[0, 'model']}__{best_selection_df.loc[0, 'feature_set']}"

display(all_comparison_df.sort_values(["model", "version", "num_features"]))
print("Best RF-selection configuration:")
display(best_selection_df)
print("Best model key:", best_model_key)

,model,version,feature_set,num_features,accuracy,precision,recall,f1_score,roc_auc
2,Random Forest,+10 EDA features,all_eda_10_features,NaN,0.643287,0.618801,0.588619,0.603333,0.695070
7,Random Forest,+10 EDA features + RF selection,top_50,50.0,0.635426,0.609162,0.583022,0.595806,0.692347
5,Random Forest,+10 EDA features + RF selection,top_75,75.0,0.639172,0.615059,0.580224,0.597134,0.693607
6,Random Forest,+10 EDA features + RF selection,top_100,100.0,0.637944,0.613198,0.580757,0.596537,0.694691
4,Random Forest,+10 EDA features + RF selection,top_150,150.0,0.639663,0.615722,0.580357,0.597517,0.695611
0,Random Forest,Baseline,all_baseline_features,NaN,0.644823,0.614810,0.614072,0.614441,0.699756
3,SVM,+10 EDA features,all_eda_10_features,NaN,0.625906,0.598659,0.571295,0.584657,0.670524
11,SVM,+10 EDA features + RF selection,top_50,50.0,0.620378,0.601660,0.521722,0.558847,0.657708
10,SVM,+10 EDA features + RF selection,top_75,75.0,0.620993,0.600271,0.531716,0.563918,0.658701
9,SVM,+10 EDA features + RF selection,top_100,100.0,0.620317,0.598775,0.533982,0.564525,0.659892


Best RF-selection configuration:


,model,feature_set,num_features,accuracy,precision,recall,f1_score,roc_auc,train_time_sec
0,Random Forest,top_150,150,0.639663,0.615722,0.580357,0.597517,0.695611,3.388638


Best model key: Random Forest__top_150


## 7. Save Outputs

In [9]:
importance_df.to_csv(OUTPUT_DIR / "rf_feature_importance.csv", index=False)
selection_results_df.to_csv(OUTPUT_DIR / "rf_selection_validation_metrics.csv", index=False)
all_comparison_df.to_csv(OUTPUT_DIR / "baseline_vs_eda10_vs_rf_selection.csv", index=False)
pd.DataFrame({"new_feature": new_features}).to_csv(OUTPUT_DIR / "new_10_features.csv", index=False)
pd.DataFrame([fe_params]).to_csv(OUTPUT_DIR / "feature_engineering_params.csv", index=False)

for feature_set_name, features in selected_feature_sets.items():
    pd.DataFrame({"feature": features}).to_csv(OUTPUT_DIR / f"selected_{feature_set_name}_features.csv", index=False)

for model_key, cm in confusion_matrices.items():
    safe_name = model_key.lower().replace(" ", "_").replace("__", "_")
    cm.to_csv(OUTPUT_DIR / f"{safe_name}_validation_confusion_matrix.csv")
    classification_reports[model_key].to_csv(OUTPUT_DIR / f"{safe_name}_validation_classification_report.csv")

joblib.dump(preprocessor, OUTPUT_DIR / "preprocessor.joblib")
joblib.dump(selector, OUTPUT_DIR / "rf_feature_selector.joblib")
joblib.dump(trained_models[best_model_key], OUTPUT_DIR / "best_rf_selection_model.joblib")

print(f"Saved outputs to: {OUTPUT_DIR}")

Saved outputs to: d:\HocTap\KT&XLTT\CUOIKI\train\feature_engineering\eda_10_rf_selection_outputs
